# OCR benchmark trên Kaggle

**Trước khi chạy**
1. Tải thư mục `ocr-bench` lên làm Kaggle Dataset (vd. tên `ocr-bench-code`) và *Add Input* vào notebook.
2. Tải bộ dữ liệu (ảnh + đáp án + `manifest.jsonl`) lên làm Dataset riêng và *Add Input*.
3. *Settings*: **Accelerator = GPU T4 x2**, **Internet = On**.
4. (Tùy chọn) *Add-ons → Secrets*: thêm `HF_TOKEN` để tải model từ Hugging Face nhanh hơn.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

## 1. Cài đặt

In [ ]:
CODE_DIR = "/kaggle/input/ocr-bench-code/ocr-bench"   # <-- sửa nếu tên Dataset khác

!rm -rf /kaggle/working/ocr-bench && cp -r {CODE_DIR} /kaggle/working/ocr-bench
!pip install -q -e "/kaggle/working/ocr-bench[teds,hf,tesseract,easyocr]"
!pip install -q -U "transformers>=4.57" accelerate peft bitsandbytes
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara > /dev/null && tesseract --list-langs | head -5

In [ ]:
# (Tùy chọn) PaddleOCR — cài riêng vì kéo theo nhiều thư viện
# !pip install -q "paddleocr>=3.3"

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Đã nạp HF_TOKEN")
except Exception as e:
    print("Không có HF_TOKEN (vẫn chạy được, chỉ tải chậm hơn):", e)

## 2. Tiếp tục từ lần chạy trước (nếu có)

Nếu đã gắn output của phiên trước làm input, copy thư mục `runs` về để tool chạy tiếp phần còn thiếu.

In [ ]:
PREV_RUNS = ""   # vd. "/kaggle/input/ocr-bench-notebook/runs" — để trống nếu chạy lần đầu
if PREV_RUNS:
    !mkdir -p /kaggle/working/runs && cp -rn {PREV_RUNS}/. /kaggle/working/runs/

## 3. Cấu hình

In [ ]:
import yaml
DATASET = "/kaggle/input/YOUR-DATASET/manifest.jsonl"   # <-- sửa

cfg = yaml.safe_load(open("/kaggle/working/ocr-bench/configs/kaggle_example.yaml"))
cfg["dataset"] = DATASET
cfg["output_dir"] = "/kaggle/working/runs"
# Bật/tắt model tại đây, ví dụ:
# for m in cfg["models"]:
#     m["enabled"] = m["name"] in {"tesseract", "baseer", "qari_0_4"}
yaml.safe_dump(cfg, open("/kaggle/working/config.yaml", "w"), allow_unicode=True, sort_keys=False)
print([m["name"] for m in cfg["models"] if m.get("enabled", True)])

In [ ]:
!ocrbench validate --config /kaggle/working/config.yaml

## 4. Chạy thử 5 mẫu

Để bắt lỗi cấu hình (sai tên model, thiếu thư viện, hết VRAM...) trước khi chạy cả bộ.

In [ ]:
!ocrbench run --config /kaggle/working/config.yaml --limit 5 --gpus 0,1

## 5. Chạy toàn bộ bộ dev

In [ ]:
!ocrbench run --config /kaggle/working/config.yaml --gpus 0,1

## 6. Xem báo cáo

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/runs/dev/_report/report.md", encoding="utf-8").read()))

In [ ]:
!cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip